# Dataleague Final Veri Okuma

Bu notebook, `datathonFINAL.parquet` dosyasini `.venv` icindeki paketlerle okuyup ilk kesfi yapmaniz icin hazirlandi.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

DATA_PATH = Path("..").resolve() / "datathonFINAL.parquet"
DATA_PATH

In [ ]:
parquet_file = pq.ParquetFile(DATA_PATH)

print(f"Dosya: {DATA_PATH.name}")
print(f"Boyut (GB): {DATA_PATH.stat().st_size / 1024**3:.2f}")
print(f"Toplam satir: {parquet_file.metadata.num_rows:,}")
print(f"Row group sayisi: {parquet_file.num_row_groups}")
print("\nKolonlar:")
print(parquet_file.schema.names)

In [ ]:
# Ilk kesif icin daha hafif bir okuma
preview_columns = [
    "original_text",
    "english_keywords",
    "sentiment",
    "main_emotion",
    "primary_theme",
    "language",
    "url",
    "author_hash",
    "date",
]

preview_df = pd.read_parquet(DATA_PATH, columns=preview_columns)
preview_df

In [ ]:
preview_df.info(memory_usage="deep")

In [ ]:
preview_df.isna().mean().sort_values(ascending=False)

In [ ]:
preview_df["language"].value_counts(dropna=False).head(20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

url_counts = preview_df["url"].fillna("NaN").value_counts(dropna=False)
sorted_counts = url_counts.sort_values(ascending=False)
ranks = np.arange(1, len(sorted_counts) + 1)
log_counts = np.log10(sorted_counts.to_numpy())

start = np.array([ranks[0], log_counts[0]], dtype=float)
end = np.array([ranks[-1], log_counts[-1]], dtype=float)
line_vec = end - start
point_vecs = np.column_stack((ranks, log_counts)) - start
distances = np.abs(np.cross(line_vec, point_vecs) / np.linalg.norm(line_vec))
break_idx = int(np.argmax(distances))
break_rank = ranks[break_idx]
break_url = sorted_counts.index[break_idx]
break_count = int(sorted_counts.iloc[break_idx])

plt.figure(figsize=(14, 6))
plt.plot(ranks, sorted_counts.to_numpy(), marker="o", linewidth=1.5, markersize=3)
plt.axvline(break_rank, color="red", linestyle="--", label=f"Break point: rank {break_rank}")
plt.scatter([break_rank], [break_count], color="red", zorder=5)
plt.yscale("log")
plt.title("URL Value Counts Break Point")
plt.xlabel("URLs sorted by frequency (rank)")
plt.ylabel("Count (log scale)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Breaking point rank: {break_rank}")
print(f"URL at breaking point: {break_url}")
print(f"Count at breaking point: {break_count}")
print(f"URLs above breaking point: {break_rank - 1}")
print(f"URLs at or below breaking point: {len(sorted_counts) - break_rank + 1}")
print("High frequency URLs above breaking point:")
print(sorted_counts.head(break_rank - 1).index.tolist())

In [ ]:
preview_df.value_counts()

In [ ]:
# Tum veri lazim olursa bu hucreyi acabilirsiniz.
# RAM siniri varsa once yukaridaki kesif hucreleriyle ilerleyin.
# full_df = pd.read_parquet(DATA_PATH)
# full_df.head()